In [10]:
from ultralytics import YOLO
import cv2
from collections import defaultdict
model = YOLO("../model/weights.pt")
import numpy as np

print(model.names)

{0: 'ball', 1: 'player', 2: 'soccer ball'}


In [11]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict

# Fine-tuned model
model = YOLO("../model/weights.pt")

print(model.names)
# {0: 'ball', 1: 'player', 2: 'soccer ball'}

VIDEO_PATH = "../data/121364_0.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

track_history = defaultdict(list)

while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break

    # Track players only
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[1],
        verbose=False
    )

    result = results[0]

    # Let Ultralytics draw the YOLO boxes + class/confidence + tracking ID
    annotated_frame = result.plot(
        line_width=1,
        font_size=0.5,
        labels=True,
        conf=True
    )

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu()
        track_ids = result.boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):

            x1, y1, x2, y2 = box

            # Bottom-center point
            x = float((x1 + x2) / 2)
            y = float(y2)

            # Store trajectory
            track_history[track_id].append((x, y))

            # Only keep recent movement
            if len(track_history[track_id]) > 20:
                track_history[track_id].pop(0)

            # Draw bottom-center point
            cv2.circle(
                annotated_frame,
                (int(x), int(y)),
                3,
                (255, 255, 255),
                -1
            )

            # Draw trajectory
            points = track_history[track_id]

            if len(points) > 1:
                pts = np.array(
                    points,
                    dtype=np.int32
                ).reshape((-1, 1, 2))

                cv2.polylines(
                    annotated_frame,
                    [pts],
                    isClosed=False,
                    color=(255, 255, 255),
                    thickness=1
                )

    cv2.imshow("Player Tracking", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

{0: 'ball', 1: 'player', 2: 'soccer ball'}
